# Food Import Risk Triage

Severity classification and inspection prioritisation for EU RASFF food import
notifications. MSc Data Science dissertation, University of Birmingham Dubai.

This notebook is a driver. The pipeline itself lives in the repository as an
importable package under `src/rasff/`, and every cell below calls into it.

Keeping the analysis in a notebook *and* in a package would mean two copies of
the same calculation, which drift apart. So there is one copy, in the package.
To see how a number is computed, open the module rather than this file.
`WHERE_THINGS_ARE.md` maps each figure in the report to the function that
produces it.

The only step that requires Colab is the DistilBERT arm in section 6, which
needs a GPU. Everything else runs on CPU.

## Before running

1. Runtime > Change runtime type > **T4 GPU**
2. Run the cells in order
3. Upload `RASFF_window.csv` when section 2 asks for it

Around 15 minutes without DistilBERT, 25 with it.

## 1. Get the code

Clone the repository and install the package. Everything after this imports
from `rasff`.

In [1]:
REPO = "https://github.com/arazack-gitrepo/food-import-risk-triage.git"

!git clone -q $REPO food-import-risk-triage || echo "CLONE FAILED - check the REPO url above and that the repository is reachable"
%cd food-import-risk-triage

# Colab ships with lightgbm. the report fits every tier with the same
# estimator so a step measures information rather than a change of algorithm,
# and lightgbm competing here would change which model wins tier 3. removing
# it makes this run match run_experiment.py on a clean machine.
!pip -q uninstall -y lightgbm 2>/dev/null

!pip -q install -e . 2>&1 | tail -2
!pip -q install tabulate 2>&1 | tail -1

import sys
sys.path.insert(0, "src")

from rasff.config import (
    RAW_CSV, YEAR_MIN, SEED, DEPLOYMENT_TIER, LABEL_ARMS,
    FEATURE_SETS, FEATURE_TIERS, ensure_output_dirs,
)
from rasff.models.zoo import available_models
ensure_output_dirs()
print("seed", SEED, "| window from", YEAR_MIN, "| deployment tier", DEPLOYMENT_TIER)
print("models:", ", ".join(available_models()))

/content/food-import-risk-triage
seed 42 | window from 2023 | deployment tier tier1_declaration
models: majority, logreg, linear_svm, random_forest


## 2. Load the export

`load_raw` reads the CSV and maps the portal's column names onto the canonical
names used throughout the package, so a change in the export format is handled
in one place.

`clean` derives `hazard_category` from the braced hazards field, parses dates
and normalises the categorical columns.

The counts printed here are the ones reported in the Data chapter: rows read,
rows repaired, rows analysed.

- Code: `src/rasff/data/loading.py`, `src/rasff/data/cleaning.py`

In [2]:
import os, shutil

if not os.path.exists(RAW_CSV):
    from google.colab import files
    print("upload RASFF_window.csv")
    up = files.upload()
    os.makedirs(RAW_CSV.parent, exist_ok=True)
    shutil.move(list(up)[0], RAW_CSV)

from rasff.data.loading import load_raw
from rasff.data.cleaning import clean

raw, load_counts = load_raw(RAW_CSV)
frame, clean_diag = clean(raw)

print("load :", load_counts)
print("clean:", clean_diag)

# no day above 12 anywhere would mean day-first vs month-first cannot be
# settled from the data alone. check the export format by hand if this fires.
if clean_diag["day_month_ambiguous"]:
    print("\nWARNING: date order is ambiguous, confirm the export format")

upload RASFF_window.csv


Saving RASFF_window.csv to RASFF_window.csv
load : {'rows_in_file': 19890, 'repaired': 12, 'dropped': 0, 'deduplicated': 0, 'rows_loaded': 19890}
clean: {'hazards_blank_pct': 26.4, 'category_coverage_pct': 73.5, 'n_categories': 28, 'unparseable_dropped': 0, 'day_month_ambiguous': False, 'date_min': '2022-01-03', 'date_max': '2025-12-31', 'duplicate_subject_pct': 14.3, 'rows': 19890}


## 3. Labels and the 2023 window

`assign_labels` maps the regulator's `risk_decision` field onto a severity
label.

`regime_tables` reports the share of each decision value by year. This is the
evidence for restricting the window to 2023 onward: the RASFF risk taxonomy
changed during 2023, so training across that boundary would fit two different
labelling policies at once and the resulting model would be answering neither
question cleanly.

`select_window` then applies the cut and drops the `undecided` rows.
`undecided` is a deprecated category from the earlier scheme rather than the
regulator recording uncertainty, so treating it as a third outcome would
misrepresent what the label means.

- Code: `src/rasff/data/labels.py`
- Report: produces the 15,331 figure and the label distribution

In [3]:
from rasff.data.labels import assign_labels, select_window, regime_tables

frame = assign_labels(frame, scheme="baseline")

regime = regime_tables(frame)
print("mapped label share by year (%):")
print(regime["label_by_year"].to_string())

analysis, window_counts = select_window(frame, YEAR_MIN)
print("\n", window_counts)

mapped label share by year (%):
label  no_risk  not_serious  serious  undecided
year                                           
2022       1.3         13.7     58.4       26.6
2023       1.6         32.1     63.4        2.9
2024       2.6         33.2     64.1        0.0
2025       1.1         34.1     64.8        0.0

 {'rows_in_window': 15469, 'undecided_dropped': 138, 'rows_analysed': 15331, 'label_counts': {'serious': 9919, 'not_serious': 5136, 'no_risk': 276}}


## 4. Splits and the experiment grid

The temporal split trains on the earliest notifications and tests on the
latest, matching how the model would be used in practice. The random split is
computed as well, but only as a comparison figure, since most published work on
RASFF reports random-split results and the difference is worth showing.

`run_grid` evaluates every combination of split, label arm, feature set and
model. The protocol is defined once, in `fit_score`: tune on validation, refit
on train and validation together, predict on test once.

The feature tiers are the substantive part of this section. Each tier adds
fields that only become knowable later in a consignment's life, so the gap
between two tiers measures what that later knowledge is worth. A model given
the complete record scores well, but some of that record does not exist at the
moment an inspector decides what to open.

- Code: `src/rasff/evaluation/splits.py`, `src/rasff/evaluation/experiment.py`
- Report: the headline table in Results

In [4]:
from rasff.evaluation.splits import make_splits, describe_split
from rasff.evaluation.experiment import run_grid, headline_table
from rasff.models.zoo import available_models

splits = make_splits(analysis, ["temporal", "random"], seed=SEED)
for s in splits.values():
    print(describe_split(s))

results_table, results = run_grid(
    splits=splits,
    arms=LABEL_ARMS,
    feature_sets=FEATURE_SETS,
    model_names=available_models(),
    seed=SEED,
)

print("\n=== temporal split, binary arm ===")
print(headline_table(results_table, "temporal", "binary_serious").to_string(index=False))

{'split': 'temporal', 'train': 10731, 'val': 2300, 'test': 2300, 'train_ends': '2025-03-03', 'test_starts': '2025-08-08', 'test_label_counts': {'serious': 1419, 'not_serious': 859, 'no_risk': 22}}
{'split': 'random', 'train': 10731, 'val': 2299, 'test': 2301, 'test_label_counts': {'serious': 1489, 'not_serious': 771, 'no_risk': 41}}
  temporal binary_serious none                         majority       macro F1 0.382 [0.374, 0.389]
  temporal binary_serious tier1_declaration            logreg         macro F1 0.666 [0.647, 0.684]
  temporal binary_serious tier1_declaration            linear_svm     macro F1 0.668 [0.648, 0.685]
  temporal binary_serious tier1_declaration            random_forest  macro F1 0.686 [0.668, 0.705]
  temporal binary_serious tier2_plus_reporter          logreg         macro F1 0.684 [0.665, 0.702]
  temporal binary_serious tier2_plus_reporter          linear_svm     macro F1 0.685 [0.666, 0.703]
  temporal binary_serious tier2_plus_reporter          random_for

/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


  temporal three_class    hybrid                       linear_svm     macro F1 0.554 [0.512, 0.597]
  temporal three_class    hybrid                       random_forest  macro F1 0.548 [0.516, 0.582]
  random   binary_serious none                         majority       macro F1 0.393 [0.386, 0.400]
  random   binary_serious tier1_declaration            logreg         macro F1 0.649 [0.629, 0.668]
  random   binary_serious tier1_declaration            linear_svm     macro F1 0.649 [0.628, 0.668]
  random   binary_serious tier1_declaration            random_forest  macro F1 0.665 [0.644, 0.684]
  random   binary_serious tier2_plus_reporter          logreg         macro F1 0.675 [0.655, 0.694]
  random   binary_serious tier2_plus_reporter          linear_svm     macro F1 0.669 [0.650, 0.688]
  random   binary_serious tier2_plus_reporter          random_forest  macro F1 0.716 [0.698, 0.735]
  random   binary_serious tier3_plus_notification_type logreg         macro F1 0.796 [0.779, 0.811]


/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


  random   three_class    hybrid                       linear_svm     macro F1 0.659 [0.611, 0.702]
  random   three_class    hybrid                       random_forest  macro F1 0.611 [0.573, 0.647]

=== temporal split, binary arm ===
                 feature_set         model  macro_f1  macro_f1_lo  macro_f1_hi  balanced_accuracy  accuracy  n_test
                        none      majority     0.382        0.374        0.389              0.500     0.617    2300
           tier1_declaration random_forest     0.686        0.668        0.705              0.690     0.699    2300
         tier2_plus_reporter random_forest     0.699        0.680        0.718              0.703     0.710    2300
tier3_plus_notification_type random_forest     0.810        0.794        0.826              0.823     0.815    2300
           tier4_plus_hazard random_forest     0.839        0.823        0.854              0.851     0.843    2300
              tier5_post_hoc        logreg     0.829        0.814   

## 5. Ablations

**Tier ablation** walks up the information ladder and shows how much of the
headline score depends on fields recorded after the inspection decision. If the
score only appears once `notification_type` is added, then the deployable
figure is the tier below it, and reporting the higher number alone would
overstate what the model knows at decision time.

**Text ablation** addresses the second research question: does the free-text
hazard description add predictive value over the structured fields available at
the same moment? It is benchmarked against tier 1, because comparing text
against a baseline that already contains post-inspection knowledge would answer
a different question.

- Code: `src/rasff/evaluation/ablation.py`
- Report: Results, and the contribution argument in Discussion

In [5]:
from rasff.evaluation.ablation import tier_ablation, text_ablation

tiers = tier_ablation(results, results_table, "temporal", "binary_serious", seed=SEED)
print("=== what each layer of information buys ===")
print(tiers.to_string(index=False))

texts = text_ablation(results, results_table, "temporal", "binary_serious", seed=SEED)
print("\n=== text vs structured ===")
print(texts.to_string(index=False))

=== what each layer of information buys ===
   split            arm                                               step                                                        adds                        available_at  macro_f1_lower  macro_f1_upper  delta  ci_lo  ci_hi  p_no_gain                                 verdict
temporal binary_serious            tier2_plus_reporter - tier1_declaration                                           notifying_country pre-inspection (RASFF network only)           0.686           0.699  0.012 -0.004  0.028      0.081 indistinguishable (interval spans zero)
temporal binary_serious tier3_plus_notification_type - tier2_plus_reporter                                           notification_type                           at filing           0.699           0.810  0.111  0.093  0.130      0.000                    significantly better
temporal binary_serious   tier4_plus_hazard - tier3_plus_notification_type                                             hazard_categ

## 6. DistilBERT

This is the only section that requires the GPU.

The model code is in `src/rasff/models/distilbert.py` rather than in this
notebook, so there is a single implementation. The protocol matches the rest of
the study: select the epoch on validation, predict on test once.

Predictions are returned as arrays and passed through
`add_external_comparison`, which applies the same paired bootstrap used for
every other comparison in the report. The DistilBERT figure is therefore
directly comparable to the structured models rather than measured on a
different footing.

- Report: RQ2, the transformer comparison

In [6]:
!pip -q install torch transformers 2>&1 | tail -1

from rasff.models.distilbert import run_distilbert
from rasff.evaluation.ablation import add_external_comparison
import pandas as pd

db_rows, db_comparisons = [], []

for arm in LABEL_ARMS:
    print(f"\n=== DistilBERT | {arm} ===")
    db = run_distilbert(splits["temporal"], arm, seed=SEED)
    db_rows.append(db.to_row())

    cmp = add_external_comparison(
        results, results_table, "temporal", arm,
        db.y_true, db.y_pred, label="distilbert", seed=SEED,
    )
    db_comparisons.append(cmp)
    print(f"  macro F1 {db.macro_f1:.3f} [{db.macro_f1_lo:.3f}, {db.macro_f1_hi:.3f}]")
    print(f"  vs tier1: {cmp['delta']:+.3f} [{cmp['ci_lo']:+.3f}, {cmp['ci_hi']:+.3f}] "
          f"-> {cmp['verdict']}")

print()
print(pd.DataFrame(db_comparisons).to_string(index=False))


=== DistilBERT | binary_serious ===
device: cuda


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch 1  loss 0.5563  val macro F1 0.7530
  epoch 2  loss 0.4059  val macro F1 0.7767
  epoch 3  loss 0.3246  val macro F1 0.7733
  epoch 4  loss 0.2617  val macro F1 0.7711
  epoch 5  loss 0.2166  val macro F1 0.7788
  best epoch 5 (val 0.7788), restoring it
  macro F1 0.777 [0.759, 0.794]
  vs tier1: +0.090 [+0.068, +0.112] -> significantly better

=== DistilBERT | three_class ===
device: cuda


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch 1  loss 0.9303  val macro F1 0.5162
  epoch 2  loss 0.7026  val macro F1 0.5437
  epoch 3  loss 0.5696  val macro F1 0.5402
  epoch 4  loss 0.4444  val macro F1 0.5433
  epoch 5  loss 0.3706  val macro F1 0.5571
  best epoch 5 (val 0.5571), restoring it
  macro F1 0.521 [0.499, 0.557]
  vs tier1: +0.057 [+0.029, +0.089] -> significantly better

   split            arm                     comparison      model baseline_model  delta  ci_lo  ci_hi  p_no_gain              verdict
temporal binary_serious distilbert - tier1_declaration distilbert  random_forest  0.090  0.068  0.112        0.0 significantly better
temporal    three_class distilbert - tier1_declaration distilbert     linear_svm  0.057  0.029  0.089        0.0 significantly better


## 7. Prioritisation under an inspection budget

Macro F1 measures classification quality. It does not answer the question a
border authority actually faces, which is: given capacity to open 10% of
consignments this week, which 10%? That is a ranking problem, so it is measured
with a ranking metric.

The model column should be read against the perfect column. At a 10% budget no
more than 10% of consignments can be opened, so if serious cases make up 16% of
the window then perfect ordering catches 16%, not 100%. Recall quoted without
that ceiling understates the result.

- Code: `src/rasff/evaluation/prioritisation.py`
- Report: the operational headline; the dashboard's priority band applies the
  same top-k rule

In [7]:
from rasff.evaluation.experiment import fit_score
from rasff.evaluation.prioritisation import (
    detection_curve, origin_risk_scores, serious_scores, summarise,
)
from rasff.data.labels import apply_arm
from rasff.models.zoo import DEPLOYMENT_MODEL
import pandas as pd

split = splits["temporal"]
deployment = fit_score(split, "binary_serious", DEPLOYMENT_TIER, DEPLOYMENT_MODEL, seed=SEED)

full_train = pd.concat([split.train, split.val], ignore_index=True)
y_train = apply_arm(full_train["label"], "binary_serious")
y_test = apply_arm(split.test["label"], "binary_serious")

curve = detection_curve(
    y_test=y_test,
    model_scores=serious_scores(deployment.fitted, split.test),
    heuristic_scores=origin_risk_scores(full_train, y_train, split.test),
    seed=SEED,
)
print(curve.to_string(index=False))
print("\nat a 10% budget:", summarise(curve, "10%"))

budget  n_inspected  model  random  worst_origin  perfect  lift_vs_random  pct_of_ceiling
    5%          115  0.078    0.05         0.068    0.081            1.55            95.7
   10%          230  0.147    0.10         0.139    0.162            1.47            90.9
   20%          460  0.285    0.20         0.271    0.324            1.43            88.0
   30%          690  0.410    0.30         0.392    0.486            1.37            84.3
   50%         1150  0.645    0.50         0.576    0.810            1.29            79.6

at a 10% budget: {'budget': '10%', 'recall': 0.147, 'ceiling': 0.162, 'pct_of_ceiling': 90.9, 'lift_vs_random': 1.47}


## 8. Save the model and the held-out window

Two artefacts are written here, both required by the dashboard.

The **model card** records the estimator, feature tier and column list held by
the saved pipeline. Without it, loading a model trained under one configuration
into code expecting another fails silently and produces plausible but wrong
output. Recording the configuration alongside the model means the mismatch is
caught at load time.

The **test window** is the set of rows the dashboard ranks. It must be the
held-out notifications rather than the full export, otherwise the percentiles
shown on screen would be computed partly over rows the model was trained on.

In [8]:
import joblib, json
from rasff.config import MODELS_DIR, PREDS_DIR
from rasff.evaluation.ablation import feature_importance_by_source

joblib.dump(deployment.fitted, MODELS_DIR / "deployment_model.joblib")

(MODELS_DIR / "deployment_model_card.json").write_text(json.dumps({
    "estimator": DEPLOYMENT_MODEL,
    "feature_tier": DEPLOYMENT_TIER,
    "columns": list(deployment.fitted.named_steps["features"].transformers_[0][2]),
    "arm": "binary_serious",
    "classes": list(deployment.fitted.classes_),
    "macro_f1": deployment.macro_f1,
    "seed": SEED,
}, indent=2))

split.test.assign(date=lambda d: d["date"].astype(str)).to_csv(
    PREDS_DIR / "test_window.csv", index=False)

print("importance by source column (%):")
print(feature_importance_by_source(deployment.fitted, DEPLOYMENT_TIER).to_string(index=False))

importance by source column (%):
          source  pct_importance
  origin_country            53.2
product_category            35.3
    product_type            11.4


## 9. Write the results out

`results/SUMMARY.md` is the source for every table in the dissertation. Figures
are generated here rather than transcribed by hand, so the report and the code
cannot disagree.

In [9]:
from rasff.reporting.tables import write_table, write_summary
from rasff.config import RESULTS_DIR
import pandas as pd

for name, table in [
    ("results_all", results_table),
    ("ablation_tiers", tiers),
    ("ablation_text", texts),
    ("prioritisation", curve),
    ("distilbert", pd.DataFrame(db_rows)),
    ("distilbert_comparison", pd.DataFrame(db_comparisons)),
]:
    write_table(table, name)

write_summary(
    {"seed": SEED, "year_min": YEAR_MIN, "deployment_tier": DEPLOYMENT_TIER,
     "dataset": {**load_counts, **window_counts}},
    {"headline, binary": headline_table(results_table, "temporal", "binary_serious"),
     "tier ablation": tiers,
     "text ablation": texts,
     "distilbert": pd.DataFrame(db_comparisons),
     "prioritisation": curve},
    RESULTS_DIR / "SUMMARY.md",
)

!cd results && zip -qr ../rasff_outputs.zip . && cd ..
from google.colab import files
files.download("rasff_outputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>